# Concept 3: Augmenting the Prompt

Takes the retrieved document from Concept 2 and stuffs it into the prompt before asking the same question again.

## Setup
Store your key in Colab **Secrets** (key icon, left sidebar) as `OPENAI_API_KEY`.

In [ ]:
!pip install -q openai numpy

In [ ]:
from openai import OpenAI
from google.colab import userdata
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

## Your Documents
Colab starts with an empty filesystem each session, so we fetch the sample docs from GitHub first.

In [ ]:
import os, urllib.request
filenames = ["01_refund.txt", "02_enrollment.txt", "03_certificate.txt", "04_mentors.txt", "05_access.txt"]
base_url = "https://raw.githubusercontent.com/Roopesht/codeexamples/main/genai/llm/704/docs/"
os.makedirs("docs", exist_ok=True)
for name in filenames:
    urllib.request.urlretrieve(base_url + name, f"docs/{name}")
docs = [open(f"docs/{name}").read() for name in filenames]

In [ ]:
import numpy as np
def embed(text):
    return client.embeddings.create(model="text-embedding-3-small", input=text).data[0].embedding
def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
query = "What's Ojasa Mirai's refund policy for its courses?"
doc_vectors = [embed(d) for d in docs]
query_vector = embed(query)
scores = [cosine(query_vector, v) for v in doc_vectors]
best = docs[scores.index(max(scores))]
prompt = f"Context:\n{best}\n\nQuestion: {query}\n\nAnswer using only the context above."
response = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}])
print(response.choices[0].message.content)

## How to Extend
Compare this answer to Concept 1's guess on the exact same question. Try removing 'using only the context above' and see if it starts blending in outside knowledge.